In [25]:
import pandas as pd
import numpy as np

# 1. Load your existing Backbone (v1)
# This file already has the 568,080 rows and subway proximity
df_v1 = pd.read_parquet('../data/processed/taxi_demand_enriched_v1.parquet')

# 2. Load and Process Weather (Central Park Station WBAN: 94728)
# We use low_memory=False because NOAA files have many columns
df_weather_raw = pd.read_csv('../data/noaa_weather/nyc_weather_hourly.csv', low_memory=False)

# 3. Create Alignment Keys (Date and Hour)
df_weather_raw['timestamp'] = pd.to_datetime(df_weather_raw['DATE'])
df_weather_raw['date'] = df_weather_raw['timestamp'].dt.date.astype(str)
df_weather_raw['hour'] = df_weather_raw['timestamp'].dt.hour

# 4. Decode TMP and convert to Fahrenheit
# Report Requirement: "Temperature in Fahrenheit only"
def get_temp_f(tmp_str):
    try:
        val = float(str(tmp_str).split(',')[0])
        if val == 9999: return np.nan # Missing data flag
        return (val / 10.0 * 9/5) + 32 
    except: return np.nan

df_weather_raw['temp_f'] = df_weather_raw['TMP'].apply(get_temp_f)

# 5. Consolidate to one record per hour
# (Handles multiple weather observations per hour)
df_weather_hourly = df_weather_raw.groupby(['date', 'hour'])['temp_f'].mean().reset_index()

# 6. JOIN: Attach Weather to the Taxi Demand Backbone
df_enriched_v2 = pd.merge(
    df_v1, 
    df_weather_hourly, 
    on=['date', 'hour'], 
    how='left'
)

# 7. Fill Gaps
# Uses Forward Fill to ensure no missing hours break the ML model
df_enriched_v2['temp_f'] = df_enriched_v2['temp_f'].ffill()

# 8. EXPORT
df_enriched_v2.to_parquet('../data/processed/taxi_demand_enriched_v2.parquet', index=False)

print(f"Success! Final Dataset Shape: {df_enriched_v2.shape}")
print(f"Columns: {df_enriched_v2.columns.tolist()}")

Success! Final Dataset Shape: (568080, 12)
Columns: ['PULocationID', 'date', 'hour', 'day_of_week', 'month', 'pickup_count', 'zone', 'borough', 'centroid_x', 'centroid_y', 'subway_proximity_km', 'temp_f']


In [26]:
# 2. Export to the processed folder
df_enriched_v2.to_parquet(
    '../data/processed/taxi_demand_master_enriched.parquet', 
    index=False, 
    engine='fastparquet'
)